In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import monotonically_increasing_id
from delta.tables import DeltaTable


#**CREATE FLAG PARAMETER**

In [0]:
dbutils.widgets.text('incremental_flag','0')

In [0]:
incremental_flag=dbutils.widgets.get('incremental_flag')


# CREATING DIMENSION MODEL


In [0]:

df_src=spark.sql('''
select distinct(Model_ID) as Model_ID, Model_Category from parquet.`abfss://silver@manishdatalake11.dfs.core.windows.net/carsales`
''')

In [0]:
df_src.display()

Model_ID,Model_Category
Mah-M167,Mah
Che-M47,Che
Toy-M205,Toy
BMW-M249,BMW
Mer-M122,Mer
Hon-M215,Hon
Nis-M82,Nis
Toy-M206,Toy
Mar-M139,Mar
Ren-M207,Ren


## dm_model Sink Initial and Incremental

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
      df_sink=spark.sql('''

        SELECT  dim_model_key, Model_ID, Model_category
        from cars_catalog.gold.dim_model
        ''')
else:
        df_sink=spark.sql('''

        SELECT 1 as dim_model_key, Model_ID, Model_category
        from PARQUET.`abfss://silver@manishdatalake11.dfs.core.windows.net/carsales`
        where 1=0
        ''')

## Filtering new records and old records


In [0]:
df_filter= df_src.join(df_sink, df_src['Model_ID']==df_sink['Model_ID'], 'left').select(df_src['Model_ID'],df_src['Model_Category'],df_sink['dim_model_key'])

In [0]:
df_filter.display()

Model_ID,Model_Category,dim_model_key
Mah-M167,Mah,1
Che-M47,Che,2
Toy-M205,Toy,3
BMW-M249,BMW,4
Mer-M122,Mer,5
Hon-M215,Hon,6
Nis-M82,Nis,7
Toy-M206,Toy,8
Mar-M139,Mar,9
Ren-M207,Ren,10


**df_filter_old**

In [0]:
df_filter_old=df_filter.filter(col('dim_model_key').isNotNull())

**df_filter_new**

In [0]:
df_filter_new=df_filter.filter(col('dim_model_key').isNull()).select('Model_ID','Model_Category')


In [0]:
df_filter_new.display()

Model_ID,Model_Category


In [0]:
df_filter_new.display()

Model_ID,Model_Category



### Create Surrogate Key


**Fetch the max surrogate key from exisitng table**

###Create Surrogate Key coulumn and ADD the max suurogate key

In [0]:
%python
if incremental_flag == '0':
    max_value = 1
else:
    max_value = spark.sql(
        "SELECT MAX(dim_model_key) FROM cars_catalog.gold.dim_model"
     ).collect()[0][0]+1

In [0]:
%python
df_filter_new = df_filter_new.withColumn('dim_model_key', monotonically_increasing_id() + max_value)



### Create Final DF - df_filter_old + df_filter_new


In [0]:
df_final = df_filter_old.union(df_filter_new)

In [0]:
df_final.display()

Model_ID,Model_Category,dim_model_key
Mah-M167,Mah,1
Che-M47,Che,2
Toy-M205,Toy,3
BMW-M249,BMW,4
Mer-M122,Mer,5
Hon-M215,Hon,6
Nis-M82,Nis,7
Toy-M206,Toy,8
Mar-M139,Mar,9
Ren-M207,Ren,10


# SCD Type (Upsert)

In [0]:
#Incremental Run
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    delta_tbl = DeltaTable.forPath(spark, "abfss://gold@manishdatalake11.dfs.core.windows.net/gold/dim_model")
    delta_tbl.alias("trg").merge(
        df_final.alias("src"), "trg.dim_model_key = src.dim_model_key") \
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()

#intial Run
else:
    df_final.write.format("delta")\
    .mode("overwrite")\
    .option("path", "abfss://gold@manishdatalake11.dfs.core.windows.net/gold/dim_model")\
    .saveAsTable("cars_catalog.gold.dim_model")


In [0]:
%sql

select * from cars_catalog.gold.dim_model;

Model_ID,Model_Category,dim_model_key
Mah-M167,Mah,1
Che-M47,Che,2
Toy-M205,Toy,3
BMW-M249,BMW,4
Mer-M122,Mer,5
Hon-M215,Hon,6
Nis-M82,Nis,7
Toy-M206,Toy,8
Mar-M139,Mar,9
Ren-M207,Ren,10
